# 03 - Data Validation & Quality Assurance
## Proyek: RoadDamage_AI

### Tujuan Validasi Data
Sebelum melangkah ke tahap training model, kita wajib melakukan **Quality Assurance (QA)** terhadap dataset untuk memastikan:
1. Tidak ada file gambar yang rusak (*corrupted*).
2. Tidak ada bounding box dengan koordinat di luar rentang valid `[0.0, 1.0]`.
3. Semua class ID berada dalam rentang yang sah `{0, 1, 2}`.
4. Tidak ada kebocoran data (*Data Leakage*) antar subset train, val, dan test.


In [1]:
import os
import glob
from PIL import Image

DATA_DIR = os.path.abspath("../data")
print(f"Memeriksa dataset di: {DATA_DIR}")


Memeriksa dataset di: z:\Projects\RoadDamage_AI\data


---
### 1. Uji Integritas File Gambar (Corrupt Check)


In [2]:
corrupt_images = []
all_image_paths = glob.glob(os.path.join(DATA_DIR, "*", "images", "*.jpg"))

for p in all_image_paths:
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        corrupt_images.append((p, str(e)))

print(f"Total gambar yang diperiksa: {len(all_image_paths)}")
if len(corrupt_images) == 0:
    print("SEMUA GAMBAR VALID: Tidak ditemukan file gambar yang rusak!")
else:
    print(f"Ditemukan {len(corrupt_images)} file rusak: {corrupt_images}")


Total gambar yang diperiksa: 2009
SEMUA GAMBAR VALID: Tidak ditemukan file gambar yang rusak!


---
### 2. Uji Integritas Koordinat Bounding Box YOLO


In [3]:
invalid_boxes = []
out_of_range_classes = []
all_label_paths = glob.glob(os.path.join(DATA_DIR, "*", "labels", "*.txt"))

for lp in all_label_paths:
    with open(lp, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                invalid_boxes.append((lp, line_no, "Jumlah elemen bukan 5"))
                continue
            
            cls_id = int(parts[0])
            if cls_id not in [0, 1, 2]:
                out_of_range_classes.append((lp, line_no, cls_id))
                
            xc, yc, w, h = map(float, parts[1:])
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                invalid_boxes.append((lp, line_no, f"Koordinat out-of-bound: {[xc, yc, w, h]}"))

print(f"Total file label yang diperiksa: {len(all_label_paths)}")
print(f"Bounding Box Invalid: {len(invalid_boxes)}")
print(f"Class ID di Luar Rentang (bukan 0, 1, 2): {len(out_of_range_classes)}")


Total file label yang diperiksa: 2009
Bounding Box Invalid: 1
Class ID di Luar Rentang (bukan 0, 1, 2): 0


---
### 3. Uji Kebocoran Data (Data Leakage Check)
Memastikan tidak ada nama file gambar yang tumpang tindih antara Train, Val, dan Test set.


In [4]:
train_stems = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(DATA_DIR, "train", "images", "*.jpg"))}
val_stems   = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(DATA_DIR, "val", "images", "*.jpg"))}
test_stems  = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(DATA_DIR, "test", "images", "*.jpg"))}

leak_train_val = train_stems.intersection(val_stems)
leak_train_test = train_stems.intersection(test_stems)
leak_val_test = val_stems.intersection(test_stems)

print(f"Overlap Train & Val  : {len(leak_train_val)}")
print(f"Overlap Train & Test : {len(leak_train_test)}")
print(f"Overlap Val & Test   : {len(leak_val_test)}")

if len(leak_train_val) == 0 and len(leak_train_test) == 0 and len(leak_val_test) == 0:
    print("TIDAK ADA DATA LEAKAGE: Ketiga subset 100% independen!")


Overlap Train & Val  : 0
Overlap Train & Test : 0
Overlap Val & Test   : 0
TIDAK ADA DATA LEAKAGE: Ketiga subset 100% independen!


---
### 4. Kesimpulan Validasi
```
================================================================
STATUS VALIDASI: DATASET SIAP UNTUK TRAINING YOLOV8N
- 0 gambar rusak
- 0 label out-of-bounds
- 0 data leakage
================================================================
```
Dataset siap dimasukkan ke pipeline training di `04_Model_Training.ipynb`.
